In [49]:
import pyuvdata
import pyuvsim 
import pyradiosky 
import fftvis
import matvis
import numpy as np
import astropy.units
from astropy.coordinates import EarthLocation
from astropy.time import Time
from astropy.units import Quantity
import time
import subprocess

### Select a single source

In [50]:
cat = pyradiosky.SkyModel()
cat.read("/fast/rbyrne/bright_sources.skyh5")

In [51]:
cat.select(min_brightness=np.max(cat.stokes))

In [52]:
print(cat.Ncomponents)

1


In [53]:
cat.write_skyh5("/fast/rbyrne/single_source.skyh5", clobber=True)

File exists; clobbering.


### Get data

In [54]:
uv = pyuvdata.UVData()
uv.read("/fast/rbyrne/20260419_055641_uvw_corrected.ms")

In [55]:
uv.select(frequencies=np.median(uv.freq_array))

In [56]:
uv.Nfreqs

1

In [57]:
uv.phase_to_time(np.mean(uv.time_array))  # Phase data
uv.flag_array = np.zeros(
    (uv.Nblts, uv.Nfreqs, uv.Npols), dtype=bool
)  # Unflag all

In [58]:
uv.write_ms("/fast/rbyrne/20260419_055641_single_freq.ms", clobber=True)

File exists; clobbering


Writing in the MS file that the units of the data are uncalib, although some CASA process will ignore this and assume the units are all in Jy (or may not know how to handle data in these units).


### Define imaging with WSClean

In [59]:
def image_data(
    name,
    ms_filepath,
    wsclean_script="/opt/bin/wsclean",
    pol="I",
    niter=0,
    weight="briggs 0",
    multiscale_scale_bias=0.8,
    size=4096,
    scale=0.03125,
    mgain=0.95,
    horizon_mask="10deg",
    auto_threshold=0.5,
    auto_mask=3,
    taper_inner_tukey=30,
    mem=30,
):
    subprocess.run(
        [
            wsclean_script,
            "-pol",
            pol,
            "-multiscale",
            "-multiscale-scale-bias",
            str(multiscale_scale_bias),
            "-size",
            str(size),
            str(size),
            "-scale",
            str(scale),
            "-niter",
            str(niter),
            "-mgain",
            str(mgain),
            "-weight",
            *weight.split(),
            "-no-update-model-required",
            "-mem",
            str(mem),
            "-horizon-mask",
            horizon_mask,
            "-auto-threshold",
            str(auto_threshold),
            "-auto-mask",
            str(auto_mask),
            "-taper-inner-tukey",
            str(taper_inner_tukey),
            "-local-rms",
            "-no-reorder",
            "-name",
            name,
            ms_filepath,
        ],
        check=True,
    )

### Run FFTvis

In [60]:
def run_fftvis_sim(data_path, model_path, beam_path, output_path, precession_and_nutation_correction=False):

    uv = pyuvdata.UVData()
    uv.read(data_path, ignore_single_chan=False)

    # Define antenna locations
    antpos = uv.telescope.get_enu_antpos()
    antpos = {a: pos for (a, pos) in zip(uv.telescope.antenna_numbers, antpos)}

    # Get observation time
    lat, lon, alt = uv.telescope.location_lat_lon_alt_degrees
    location = EarthLocation.from_geodetic(lat=lat, lon=lon, height=alt)
    obstimes = Time(
        sorted(list(set(uv.time_array))), format="jd", scale="utc", location=location
    )

    # Get beam
    uvb = pyuvdata.UVBeam.from_file(beam_path)
    uvb.peak_normalize()
    if uvb.feed_array[0].lower() == "e":
        pol_mapping = {
            v: k
            for k, v in pyuvdata.utils._x_orientation_rep_dict(
                uvb.x_orientation
            ).items()
        }
        uvb.feed_array = np.array(
            [pol_mapping[feed.lower()] for feed in uvb.feed_array]
        )
    if np.max(Quantity(uv.freq_array, "Hz")) > np.max(Quantity(uvb.freq_array, "Hz")):
        print(
            "WARNING: Max data frequency exceeds max beam model frequency. Using nearest neighbor value."
        )
        uvb_max_freq = uvb.select(frequencies=np.max(uvb.freq_array), inplace=False)
        uvb.freq_array = np.append(uvb.freq_array, np.max(uvd.freq_array))
        uvb.Nfreqs += 1
        uvb.data_array = np.append(
            uvb.data_array,
            uvb_max_freq.data_array,
            axis=2,
        )
        uvb.bandpass_array = np.append(uvb.bandpass_array, uvb_max_freq.bandpass_array)
    if np.min(Quantity(uv.freq_array, "Hz")) < np.min(Quantity(uvb.freq_array, "Hz")):
        print(
            "WARNING: Minimum data frequency is less than minimum beam model frequency. Using nearest neighbor value."
        )
        uvb_min_freq = uvb.select(frequencies=np.min(uvb.freq_array), inplace=False)
        uvb.freq_array = np.append(np.min(uv.freq_array), uvb.freq_array)
        uvb.Nfreqs += 1
        uvb.data_array = np.append(
            uvb_min_freq.data_array,
            uvb.data_array,
            axis=2,
        )
        uvb.bandpass_array = np.append(uvb_min_freq.bandpass_array, uvb.bandpass_array)
    uvb.freq_interp_kind = "linear"  # Added for simulation speedup
    uvb.interp(freq_array=uv.freq_array)

    # Get model
    if model_path.endswith(".skyh5"):
        model = pyradiosky.SkyModel.from_skyh5(model_path)
    elif model_path.endswith(".sav"):
        model = pyradiosky.SkyModel.from_fhd_catalog(model_path, expand_extended=True)
    else:
        model = pyradiosky.SkyModel()
        model.read(model_path)
    use_model_freq_array = np.copy(uv.freq_array)
    if (
        model.spectral_type == "subband"
    ):  # Define frequency extrapolation to use nearest neighbor values
        if np.max(Quantity(uv.freq_array, "Hz")) > np.max(
            Quantity(model.freq_array, "Hz")
        ):
            print(
                "WARNING: Max data frequency exceeds max sky model frequency. Using nearest neighbor value."
            )
            use_model_freq_array[
                np.where(
                    Quantity(use_model_freq_array, "Hz")
                    > np.max(Quantity(model.freq_array, "Hz"))
                )
            ] = np.max(model.freq_array)
        if np.min(Quantity(uvd.freq_array, "Hz")) < np.min(
            Quantity(model.freq_array, "Hz")
        ):
            print(
                "WARNING: Minimum data frequency is less than minimum sky model frequency. Using nearest neighbor value."
            )
            use_model_freq_array[
                np.where(
                    Quantity(use_model_freq_array, "Hz")
                    < np.min(Quantity(model.freq_array, "Hz"))
                )
            ] = np.min(model.freq_array)
    model.at_frequencies(Quantity(use_model_freq_array, "Hz"))
    if model.component_type == "healpix":
        model.healpix_to_point()
    use_comp_inds = np.where(np.isfinite(np.sum(model.stokes, axis=(0, 1))))[0]
    if len(use_comp_inds) < model.Ncomponents:  # Remove nan-ed sources
        print("Removing nan flux values from input catalog.")
        model.select(component_inds=use_comp_inds)
    
    if precession_and_nutation_correction:
        ra_new, dec_new = matvis.coordinates.equatorial_to_eci_coords(
            model.ra.rad,
            model.dec.rad,
            np.mean(obstimes),
            location,
            unit="rad",
            frame="icrs",
        )
    else:
        ra_new = model.ra.rad
        dec_new = model.dec.rad

    # Run simulation
    sim_start_time = time.time()

    antpairs = [
        (a1, a2)
        for ant1_ind, a1 in enumerate(uv.telescope.antenna_numbers)
        for a2 in uv.telescope.antenna_numbers[ant1_ind:]
    ]
    vis_full = np.zeros(
        (uv.Nfreqs, uv.Ntimes, 2, 2, len(antpairs)), dtype=complex
    )
    for freq_ind in range(uv.Nfreqs):
        vis_full[freq_ind, :, :, :, :] = fftvis.simulate_vis(
            telescope_loc=location,
            ants=antpos,
            baselines=antpairs,
            fluxes=model.stokes[0].T.to_value("Jy"),
            ra=ra_new,
            dec=dec_new,
            freqs=uv.freq_array[[freq_ind]],
            times=obstimes,
            beam=uvb,
            polarized=True,
            precision=2,
        )
    vis_full = vis_full.transpose([4, 1, 0, 2, 3]).reshape(
        (len(antpairs) * uv.Ntimes, uv.Nfreqs, 4)
    )

    sim_duration = time.time() - sim_start_time

    formatting_start_time = time.time()
    uvd_out = pyuvdata.UVData.new(
        freq_array=uv.freq_array,
        polarization_array=[-5, -7, -8, -6],
        telescope=uv.telescope,
        times=np.array(obstimes.jd),
        antpairs=np.array(antpairs),
        time_axis_faster_than_bls=True,
        data_array=vis_full,
    )

    uvd_out.reorder_blts()
    uvd_out.reorder_pols(order="AIPS")
    uvd_out.reorder_freqs(channel_order="freq")
    uvd_out.phase_to_time(np.mean(uv.time_array))
    uvd_out.check()

    # Save output
    uvd_out.reorder_pols(order="CASA")
    uvd_out.write_ms(output_path, clobber=True, fix_autos=True)

In [61]:
run_fftvis_sim("/fast/rbyrne/20260419_055641_single_freq.ms", "/fast/rbyrne/single_source.skyh5", "/fast/rbyrne/OVRO_LWA_MROsoil_updatedheight.fits", "/fast/rbyrne/single_source_fftvis_nocorrection.ms", precession_and_nutation_correction=True)
run_fftvis_sim("/fast/rbyrne/20260419_055641_single_freq.ms", "/fast/rbyrne/single_source.skyh5", "/fast/rbyrne/OVRO_LWA_MROsoil_updatedheight.fits", "/fast/rbyrne/single_source_fftvis_nocorrection.ms", precession_and_nutation_correction=False)

integration_time not provided, and cannot be inferred from time_array, setting to 1 second
channel_width not provided, and cannot be inferred from freq_array, setting to 1 Hz
Fixing auto-correlations to be be real-only, after some imaginary values were detected in data_array. Largest imaginary component was 2.7155738885477394e-14, largest imaginary/real ratio was 2.161132873875278e-17.
Writing in the MS file that the units of the data are uncalib, although some CASA process will ignore this and assume the units are all in Jy (or may not know how to handle data in these units).


File exists; clobbering


integration_time not provided, and cannot be inferred from time_array, setting to 1 second
channel_width not provided, and cannot be inferred from freq_array, setting to 1 Hz


File exists; clobbering


Fixing auto-correlations to be be real-only, after some imaginary values were detected in data_array. Largest imaginary component was 3.1376096812107586e-14, largest imaginary/real ratio was 2.4384021745963194e-17.
Writing in the MS file that the units of the data are uncalib, although some CASA process will ignore this and assume the units are all in Jy (or may not know how to handle data in these units).


In [62]:
image_data("/fast/rbyrne/single_source_fftvis", "/fast/rbyrne/single_source_fftvis.ms")
image_data("/fast/rbyrne/single_source_fftvis_nocorrection", "/fast/rbyrne/single_source_fftvis_nocorrection.ms")


WSClean version 3.4 (2023-10-11)
This software package is released under the GPL version 3.
Author: André Offringa (offringa@gmail.com).

No corrected data in first measurement set: tasks will be applied on the data column.
=== IMAGING TABLE ===
       # Pol Ch JG ²G FG FI In Freq(MHz)
| Independent group:
+-+-J- 0  I   0  0  0  0  0  0  45-45 (1)
 == Constructing image ==
Precalculating weights for Briggs'(0) weighting...
Opening /fast/rbyrne/single_source_fftvis.ms, spw 0 with contiguous MS reader.
Mapping measurement set rows... DONE (0-62128; 61776 rows)
Detected 503.2 GB of system memory, usage limited to 151 GB (frac=30%, no abs limit)
Opening /fast/rbyrne/single_source_fftvis.ms, spw 0 with contiguous MS reader.
Mapping measurement set rows... DONE (0-62128; 61776 rows)
Determining min and max w & theoretical beam size... DONE (w=[3.52649e-08:3.38828] lambdas, maxuvw=353.834 lambda)
Theoretic beam = 9.72'
Minimal inversion size: 1898 x 1898, using optimal: 1920 x 1920
Loading d

### Run pyuvsim

In [63]:
def run_pyuvsim(data_path, model_path, beam_path, output_path):
    uv = pyuvdata.UVData()
    uv.read(data_path, ignore_single_chan=False)
    
    catalog = pyradiosky.SkyModel()
    catalog.read(model_path)
    catalog_formatted = pyuvsim.simsetup.SkyModelData(catalog)

    beam = pyuvdata.UVBeam()
    beam.read(beam_path)
    beam.peak_normalize()
    beam_list = pyuvsim.BeamList(beam_list=[beam])

    output_uv = pyuvsim.uvsim.run_uvdata_uvsim(
        input_uv=uv,
        beam_list=beam_list,
        beam_dict=None,  # Same beam for all ants
        catalog=catalog_formatted,
        quiet=False,
    )
    output_uv.phase_to_time(np.mean(output_uv.time_array))
    output_uv.write_ms(output_path, clobber=True, fix_autos=True)

In [64]:
run_pyuvsim("/fast/rbyrne/20260419_055641_single_freq.ms", "/fast/rbyrne/single_source.skyh5", "/fast/rbyrne/OVRO_LWA_MROsoil_updatedheight.fits", "/fast/rbyrne/single_source_pyuvsim.ms")

Nbls: 62128
Ntimes: 1
Nfreqs: 1
Nsrcs: 1
Tasks:  62128.0
1.00% completed. 0:00:01.576346  elapsed. 0:02:35.875752 remaining. 

2.00% completed. 0:00:01.745121  elapsed. 0:01:25.409949 remaining. 

3.00% completed. 0:00:01.914172  elapsed. 0:01:01.817716 remaining. 

4.00% completed. 0:00:02.083111  elapsed. 0:00:49.934371 remaining. 

5.01% completed. 0:00:02.254604  elapsed. 0:00:42.785281 remaining. 

6.01% completed. 0:00:02.421477  elapsed. 0:00:37.889758 remaining. 

7.01% completed. 0:00:02.588135  elapsed. 0:00:34.342430 remaining. 

8.01% completed. 0:00:02.753751  elapsed. 0:00:31.628294 remaining. 

9.01% completed. 0:00:02.919500  elapsed. 0:00:29.481838 remaining. 

10.01% completed. 0:00:03.085680  elapsed. 0:00:27.735399 remaining. 

11.01% completed. 0:00:03.251710  elapsed. 0:00:26.275073 remaining. 

12.01% completed. 0:00:03.417811  elapsed. 0:00:25.030977 remaining. 

13.02% completed. 0:00:03.584838  elapsed. 0:00:23.958916 remaining. 

14.02% completed. 0:00:03.916

The parameter `blt_order` could not be identified for input_uv  so the ordering cannot be restored.The output UVData object will have (time, baseline) ordering.
Fixing auto-correlations to be be real-only, after some imaginary values were detected in data_array. Largest imaginary component was 6.394884621840902e-14, largest imaginary/real ratio was 4.627884223626953e-17.
Writing in the MS file that the units of the data are uncalib, although some CASA process will ignore this and assume the units are all in Jy (or may not know how to handle data in these units).


In [65]:
image_data("/fast/rbyrne/single_source_pyuvsim", "/fast/rbyrne/single_source_pyuvsim.ms")


WSClean version 3.4 (2023-10-11)
This software package is released under the GPL version 3.
Author: André Offringa (offringa@gmail.com).

No corrected data in first measurement set: tasks will be applied on the data column.
=== IMAGING TABLE ===
       # Pol Ch JG ²G FG FI In Freq(MHz)
| Independent group:
+-+-J- 0  I   0  0  0  0  0  0  45-45 (1)
 == Constructing image ==
Precalculating weights for Briggs'(0) weighting...
Opening /fast/rbyrne/single_source_pyuvsim.ms, spw 0 with contiguous MS reader.
Mapping measurement set rows... DONE (0-62128; 61776 rows)
Detected 503.2 GB of system memory, usage limited to 151 GB (frac=30%, no abs limit)
Opening /fast/rbyrne/single_source_pyuvsim.ms, spw 0 with contiguous MS reader.
Mapping measurement set rows... DONE (0-62128; 61776 rows)
Determining min and max w & theoretical beam size... DONE (w=[3.52649e-08:3.38828] lambdas, maxuvw=353.834 lambda)
Theoretic beam = 9.72'
Minimal inversion size: 1898 x 1898, using optimal: 1920 x 1920
Loading